In [7]:
import os
from minio import Minio
from minio.error import S3Error
import time

# Configurações do Kaggle
os.environ['KAGGLE_USERNAME'] = 'jesuinoaraujo'
os.environ['KAGGLE_KEY'] = 'bce72cd0f618c96edc8458692508173e'

# Configurações do MinIO
MINIO_ENDPOINT = "192.168.15.200:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"

# Cria um cliente MinIO
client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

# Criação dos buckets
buckets = ['raw', 'bronze', 'silver', 'gold']

for bucket in buckets:
    try:
        if not client.bucket_exists(bucket):
            client.make_bucket(bucket)
            print(f"Bucket '{bucket}' criado com sucesso!")
        else:
            print(f"Bucket '{bucket}' já existe.")
    except S3Error as err:
        print(f"Erro ao criar o bucket '{bucket}': {err}")

# Adiciona um atraso para garantir que os buckets sejam criados
time.sleep(5)

# Download e descompactação dos arquivos do Kaggle
!kaggle datasets download -d utkarshx27/non-alcohol-fatty-liver-disease -p D:/jesui/Documents/projeto-mlops/notebook

import zipfile

# Caminho do arquivo zip
zip_path = "D:/jesui/Documents/projeto-mlops/notebook/non-alcohol-fatty-liver-disease.zip"
extract_path = "D:/jesui/Documents/projeto-mlops/notebook/non-alcohol-fatty-liver-disease/"

# Descompacta o arquivo zip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Arquivos descompactados com sucesso!")

# Upload dos arquivos para o bucket 'raw'
import glob

files = glob.glob(extract_path + "*")

for file_path in files:
    file_name = os.path.basename(file_path)
    try:
        client.fput_object('raw', file_name, file_path)
        print(f"Arquivo '{file_name}' enviado para o bucket 'raw'.")
    except S3Error as err:
        print(f"Erro ao enviar o arquivo '{file_name}': {err}")

Bucket 'raw' criado com sucesso!
Bucket 'bronze' criado com sucesso!
Bucket 'silver' criado com sucesso!
Bucket 'gold' criado com sucesso!
Dataset URL: https://www.kaggle.com/datasets/utkarshx27/non-alcohol-fatty-liver-disease
License(s): CC0-1.0
non-alcohol-fatty-liver-disease.zip: Skipping, found more recently modified local copy (use --force to force download)
Arquivos descompactados com sucesso!
Arquivo 'nafld1.csv' enviado para o bucket 'raw'.
Arquivo 'nafld2.csv' enviado para o bucket 'raw'.
Arquivo 'nwtco.csv' enviado para o bucket 'raw'.
